# Supercharged LLM Chatbot
This upgraded notebook uses **Qwen 7B** (loaded in 4-bit for free T4 GPUs), real-time streaming, and a beautiful advanced interface.

## 1. Install Dependencies
We need `gradio` for the UI, `transformers` and `accelerate` for the model, and `bitsandbytes` for 4-bit quantization.

In [ ]:
!pip install -qU gradio transformers accelerate bitsandbytes

## 2. Load the Model & Tokenizer
We load `Qwen/Qwen2.5-7B-Instruct` in 4-bit precision so it fits easily in Colab's 15GB T4 GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread
import gradio as gr

model_name = "Qwen/Qwen2.5-7B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model in 4-bit precision (this may take a couple of minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16,
)
print("Model loaded successfully!")

## 3. Define the Streaming Chat Function
This function runs the model and streams the output word-by-word back to the UI.

In [ ]:
def chat(message, history, system_prompt, temperature, max_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
    ]
    
    # Add conversation history
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
        
    # Add the current user message
    messages.append({"role": "user", "content": message})
    
    # Format the messages using the model's chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize input
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Initialize the streamer
    streamer = TextIteratorStreamer(tokenizer, timeout=10., skip_prompt=True, skip_special_tokens=True)
    
    # Set up generation arguments
    generate_kwargs = dict(
        model_inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        temperature=temperature,
        do_sample=True if temperature > 0 else False,
        top_p=0.9,
    )
    
    # Start generation in a separate thread so it doesn't block
    t = Thread(target=model.generate, kwargs=generate_kwargs)
    t.start()
    
    # Yield text as it generates
    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        yield partial_text

## 4. Build and Launch the Premium UI
This creates the Gradio interface with custom CSS, avatars, and an advanced settings panel.

In [ ]:
css = """
.gradio-container {
    max-width: 950px !important;
    margin: auto !important;
    font-family: 'Inter', sans-serif;
}
#header {
    text-align: center;
    padding: 25px;
    background: linear-gradient(135deg, #6e8efb, #a777e3);
    color: white;
    border-radius: 16px;
    margin-bottom: 20px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.1);
}
#header h1 {
    font-size: 38px;
    margin: 0;
    font-weight: bold;
}
#header p {
    font-size: 16px;
    opacity: 0.95;
    margin-top: 5px;
}
.advanced-settings {
    border: 1px solid #e0e0e0;
    border-radius: 12px;
    padding: 15px;
    background-color: #f9f9f9;
}
"""

with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:
    gr.HTML('''
    <div id="header">
        <h1>🚀 Supercharged AI Chatbot</h1>
        <p>Powered by Qwen 7B • Real-time Streaming • Advanced Controls</p>
    </div>
    ''')
    
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=550, 
                bubble_full_width=False, 
                avatar_images=(None, "https://cdn-icons-png.flaticon.com/512/4712/4712027.png")
            )
            with gr.Row():
                msg = gr.Textbox(placeholder="Type your message here...", container=False, scale=7, lines=2)
                submit_btn = gr.Button("Send", variant="primary", scale=1)
                
            clear_btn = gr.Button("🗑️ Clear Conversation", size="sm")
                
        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Advanced Settings", open=True, elem_classes="advanced-settings"):
                system_prompt = gr.Textbox(
                    value="You are a helpful, smart, and friendly AI assistant.", 
                    label="System Prompt", 
                    lines=4
                )
                temperature = gr.Slider(minimum=0.0, maximum=2.0, value=0.7, step=0.1, label="Temperature")
                max_tokens = gr.Slider(minimum=1, maximum=4096, value=1024, step=64, label="Max Tokens")

    # Logic to handle message submitting
    def user_msg(user_message, history):
        return "", history + [[user_message, None]]

    def bot_response(history, sys_prompt, temp, max_tok):
        user_message = history[-1][0]
        history[-1][1] = ""
        for chunk in chat(user_message, history[:-1], sys_prompt, temp, max_tok):
            history[-1][1] = chunk
            yield history

    # Event listeners
    msg.submit(user_msg, [msg, chatbot], [msg, chatbot], queue=False).then(
        bot_response, [chatbot, system_prompt, temperature, max_tokens], chatbot
    )
    submit_btn.click(user_msg, [msg, chatbot], [msg, chatbot], queue=False).then(
        bot_response, [chatbot, system_prompt, temperature, max_tokens], chatbot
    )
    clear_btn.click(lambda: None, None, chatbot, queue=False)

# Launch the app with a queue enabled (required for generators/streaming)
demo.queue()
demo.launch(debug=True, share=True)
